# 📖 Notebook 3: Hybrid Search

In real applications, vector search alone isn't enough. Users don't just want "similar products" — they want "similar products **under $50 that are in stock**." This notebook shows how to combine vector similarity with traditional SQL filters.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why vector-only search produces bad results in practice
- **BAD**: Vector-only search ignores business rules
- **BETTER**: Pre-filter then vector search
- **BEST**: Combined scoring with partial indexes and weighted ranking

## 🛠️ Setup

```bash
cd deep-dives/vector-databases
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import numpy as np
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "vector_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

# Verify products are loaded
conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM products")
count = cur.fetchone()[0]
print(f"✅ Connected — {count} products loaded")

# Show product distribution
cur.execute("""
    SELECT category, COUNT(*), ROUND(AVG(price)::numeric, 2) AS avg_price
    FROM products
    GROUP BY category
    ORDER BY category
""")
print(f"\n{'Category':<15} {'Count':>6} {'Avg Price':>10}")
print("-" * 35)
for cat, cnt, avg_p in cur.fetchall():
    print(f"{cat:<15} {cnt:>6} ${avg_p:>8}")
conn.close()

## 🎯 The Problem: Why Vector Search Alone Fails

Imagine you're building a product search for an e-commerce site. A user searches for "wireless audio gear." Vector search will find products with similar embeddings. But what if:

- The top result is **out of stock**?
- The results include products from the **wrong category**?
- The user has a **budget of $100**?

Vector search alone doesn't know about these business constraints. Let's see this in action.

In [ ]:
# Get the embedding for "Wireless Noise-Cancelling Headphones" as our query
# Imagine a user looking for something similar to this product

conn = get_conn()
cur = conn.cursor()

cur.execute("""
    SELECT embedding FROM products
    WHERE name = 'Wireless Noise-Cancelling Headphones'
""")
query_embedding = cur.fetchone()[0]

print("🎯 Query: 'Find products similar to Wireless Noise-Cancelling Headphones'")
print("   (User wants: audio gear, budget $100, must be in stock)")
print()
conn.close()

---

## ❌ BAD: Vector-Only Search

Just find the K nearest vectors. Ignore everything else.

In [ ]:
# BAD: Pure vector search — no filters

conn = get_conn()
cur = conn.cursor()

cur.execute("""
    SELECT name, category, price, rating, in_stock,
           embedding <=> %s::vector AS distance
    FROM products
    WHERE name != 'Wireless Noise-Cancelling Headphones'
    ORDER BY embedding <=> %s::vector
    LIMIT 10
""", (query_embedding, query_embedding))

print("❌ BAD: Vector-Only Search (Top 10 by similarity)")
print("=" * 85)
print(f"{'Product':<40} {'Category':<12} {'Price':>7} {'Stock':>6} {'Dist':>6}")
print("-" * 85)

bad_results = cur.fetchall()
problems = 0
for name, cat, price, rating, stock, dist in bad_results:
    flag = ""
    if not stock:
        flag += " ⚠️  OUT OF STOCK"
        problems += 1
    if price > 100:
        flag += " ⚠️  OVER BUDGET"
        problems += 1
    if cat != 'Electronics':
        flag += " ⚠️  WRONG CATEGORY"
        problems += 1
    stock_str = "✅" if stock else "❌"
    print(f"{name:<40} {cat:<12} ${price:>6} {stock_str:>6} {dist:>6.4f}{flag}")

conn.close()

print(f"\n⚠️  {problems} problems found!")
print("   Vector search finds 'similar' vectors but ignores business rules.")
print("   Out-of-stock items, wrong categories, and overpriced products slip through.")

---

## ✅ BETTER: Pre-Filter Then Vector Search

The simplest fix: add `WHERE` clauses to filter first, then sort by vector similarity.

```sql
SELECT * FROM products
WHERE category = 'Electronics'        -- filter by category
  AND price <= 100                     -- filter by budget
  AND in_stock = true                  -- only available items
ORDER BY embedding <=> query_vector    -- then rank by similarity
LIMIT 10
```

This works! But there are tradeoffs...

In [ ]:
# BETTER: Pre-filter then vector search

conn = get_conn()
cur = conn.cursor()

cur.execute("""
    SELECT name, category, price, rating, in_stock,
           embedding <=> %s::vector AS distance
    FROM products
    WHERE category = 'Electronics'
      AND price <= 100
      AND in_stock = true
      AND name != 'Wireless Noise-Cancelling Headphones'
    ORDER BY embedding <=> %s::vector
    LIMIT 10
""", (query_embedding, query_embedding))

print("✅ BETTER: Pre-Filter + Vector Search")
print("   Filters: category='Electronics', price<=100, in_stock=true")
print("=" * 80)
print(f"{'Product':<40} {'Category':<12} {'Price':>7} {'Rating':>7} {'Dist':>6}")
print("-" * 80)

better_results = cur.fetchall()
for name, cat, price, rating, stock, dist in better_results:
    print(f"{name:<40} {cat:<12} ${price:>6} {float(rating):>6.1f} {dist:>6.4f}")

conn.close()

print(f"\n✅ All results match our filters!")
print("\n⚠️  But there's a problem: strict filters might eliminate good results.")
print("   What if a $105 product is a much better vector match than a $50 one?")
print("   Pre-filtering is binary: it either passes or gets thrown out entirely.")

### The Pre-Filtering Problem

Pre-filtering works well when filters are broad. But strict filters can:

1. **Eliminate great matches** — A product at $105 (just over budget) might be the best semantic match
2. **Return too few results** — If filters are very specific, you might not have enough vectors left
3. **Cause index issues** — If filtering removes most rows, the vector index may not be used efficiently

We need something more flexible: **weighted scoring** that considers both similarity AND metadata.

---

## 🏆 BEST: Combined Scoring with Weighted Ranking

Instead of hard filters for everything, use a **scoring function** that combines:
- **Vector similarity** (how semantically close is this?)
- **Metadata relevance** (does it match the user's preferences?)

```
final_score = α × vector_similarity + (1-α) × metadata_score
```

Where `α` (alpha) controls the balance. Typically 0.5–0.8 for vector weight.

Some filters are still hard (must be in stock), but others become soft preferences (price is a factor, not a cutoff).

In [ ]:
# BEST: Combined scoring with weighted ranking

conn = get_conn()
cur = conn.cursor()

# Hard filters: must be in stock (non-negotiable business rule)
# Soft scoring: vector similarity + price preference + rating boost

target_price = 100.0  # preferred price point
alpha = 0.6  # weight for vector similarity (0.6 = 60% vector, 40% metadata)

cur.execute("""
    WITH scored AS (
        SELECT
            name, category, subcategory, price, rating, review_count, in_stock,
            1 - (embedding <=> %s::vector) AS vector_score,
            1 - LEAST(ABS(price - %s) / %s, 1.0) AS price_score,
            rating / 5.0 AS rating_score,
            LEAST(LOG(review_count + 1) / LOG(5000), 1.0) AS popularity_score
        FROM products
        WHERE in_stock = true
          AND name != 'Wireless Noise-Cancelling Headphones'
    )
    SELECT
        name, category, price, rating, review_count,
        ROUND(vector_score::numeric, 4) AS vec_score,
        ROUND(price_score::numeric, 4) AS price_sc,
        ROUND(rating_score::numeric, 4) AS rating_sc,
        ROUND((
            %s * vector_score +
            %s * (0.4 * price_score + 0.3 * rating_score + 0.3 * popularity_score)
        )::numeric, 4) AS final_score
    FROM scored
    ORDER BY (
        %s * vector_score +
        %s * (0.4 * price_score + 0.3 * rating_score + 0.3 * popularity_score)
    ) DESC
    LIMIT 10
""", (query_embedding, target_price, target_price, alpha, 1-alpha, alpha, 1-alpha))

print("🏆 BEST: Combined Scoring (alpha=0.6)")
print("   Hard filter: in_stock=true")
print("   Soft scoring: 60% vector similarity + 40% (price + rating + popularity)")
print("=" * 95)
print(f"{'Product':<35} {'Category':<12} {'Price':>6} {'Vec':>6} {'Price':>6} {'Rate':>6} {'Final':>6}")
print("-" * 95)

best_results = cur.fetchall()
for name, cat, price, rating, reviews, vs, ps, rs, fs in best_results:
    print(f"{name:<35} {cat:<12} ${float(price):>5.0f} {float(vs):>6.4f} {float(ps):>6.4f} {float(rs):>6.4f} {float(fs):>6.4f}")

conn.close()

print()
print("💡 Notice: The ranking now considers BOTH semantic similarity AND business factors.")
print("   A slightly less similar product with a great price/rating can rank higher.")

In [ ]:
# Let's see how different alpha values change the results

conn = get_conn()
cur = conn.cursor()

print("📊 How Alpha (vector weight) Affects Rankings")
print("=" * 70)

for alpha in [0.3, 0.5, 0.7, 0.9]:
    cur.execute("""
        WITH scored AS (
            SELECT
                name, category, price,
                1 - (embedding <=> %s::vector) AS vector_score,
                1 - LEAST(ABS(price - %s) / %s, 1.0) AS price_score,
                rating / 5.0 AS rating_score,
                LEAST(LOG(review_count + 1) / LOG(5000), 1.0) AS popularity_score
            FROM products
            WHERE in_stock = true
              AND name != 'Wireless Noise-Cancelling Headphones'
        )
        SELECT name, category, price
        FROM scored
        ORDER BY (
            %s * vector_score +
            %s * (0.4 * price_score + 0.3 * rating_score + 0.3 * popularity_score)
        ) DESC
        LIMIT 3
    """, (query_embedding, 100.0, 100.0, alpha, 1-alpha))

    label = 'mostly vector' if alpha > 0.5 else ('mostly metadata' if alpha < 0.5 else 'balanced')
    results = cur.fetchall()
    print(f"\n  alpha={alpha} ({label}):")
    for name, cat, price in results:
        print(f"    -> {name} ({cat}, ${float(price):.2f})")

conn.close()

print()
print("💡 Low alpha  -> results favor cheap, highly-rated products")
print("   High alpha -> results favor semantically similar products")
print("   Tune alpha based on your use case!")

---

## ⚡ Making Hybrid Search Fast: Partial Indexes

For large datasets, we can create **partial vector indexes** — indexes that only cover a subset of rows. This makes both filtering and vector search fast:

```sql
-- Index only in-stock Electronics products
CREATE INDEX idx_electronics_hnsw
ON products USING hnsw (embedding vector_cosine_ops)
WHERE category = 'Electronics' AND in_stock = true;
```

PostgreSQL can use this index when your WHERE clause matches the index condition.

In [ ]:
# Create partial vector indexes for common filter combinations

conn = get_conn()
cur = conn.cursor()

categories = ['Electronics', 'Clothing', 'Home', 'Books', 'Sports']

print("⚡ Creating Partial HNSW Indexes")
print("=" * 50)

for cat in categories:
    index_name = f"idx_{cat.lower()}_hnsw"
    cur.execute(f"DROP INDEX IF EXISTS {index_name}")
    cur.execute(
        f"CREATE INDEX {index_name} "
        f"ON products USING hnsw (embedding vector_cosine_ops) "
        f"WHERE category = '{cat}' AND in_stock = true"
    )
    print(f"  ✅ Created {index_name}")

conn.commit()

# Now query with filters that match a partial index
cur.execute("""
    EXPLAIN ANALYZE
    SELECT name, embedding <=> %s::vector AS distance
    FROM products
    WHERE category = 'Electronics' AND in_stock = true
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (query_embedding, query_embedding))

print("\n📋 Query Plan (with partial index):")
for row in cur.fetchall():
    print(f"  {row[0]}")

conn.close()

print()
print("💡 Partial indexes are powerful for hybrid search:")
print("   The WHERE clause matches the index condition")
print("   PostgreSQL uses the partial HNSW index for fast vector search")
print("   Only the filtered subset is indexed -> less memory, faster builds")

---

## 🔤 Bonus: Full-Text Search + Vector Search

Another powerful hybrid: combine PostgreSQL's built-in **full-text search** (keyword matching) with **vector similarity** (semantic matching).

- Full-text search finds exact keyword matches: "wireless headphones" → must contain those words
- Vector search finds semantic matches: "wireless headphones" → also finds "Bluetooth earbuds"

Combining both gives the best of both worlds.

In [ ]:
# Full-text search + vector search

conn = get_conn()
cur = conn.cursor()

# Add a tsvector column for full-text search
cur.execute("ALTER TABLE products ADD COLUMN IF NOT EXISTS search_text tsvector")
cur.execute("""
    UPDATE products
    SET search_text = to_tsvector('english', name || ' ' || description)
""")
cur.execute("CREATE INDEX IF NOT EXISTS idx_products_fts ON products USING gin(search_text)")
conn.commit()

search_tsquery = "wireless & audio"

# Method 1: Full-text only
cur.execute("""
    SELECT name, category, price,
           ts_rank(search_text, to_tsquery('english', %s)) AS text_rank
    FROM products
    WHERE search_text @@ to_tsquery('english', %s)
    ORDER BY text_rank DESC
    LIMIT 5
""", (search_tsquery, search_tsquery))

print("🔤 Full-Text Search Only: 'wireless & audio'")
print("-" * 60)
fts_results = cur.fetchall()
for name, cat, price, rank in fts_results:
    print(f"  {name:<40} (rank: {rank:.4f})")
if not fts_results:
    print("  (no exact keyword matches found)")

# Method 2: Vector only
cur.execute("""
    SELECT name, category, price,
           embedding <=> %s::vector AS distance
    FROM products
    ORDER BY embedding <=> %s::vector
    LIMIT 5
""", (query_embedding, query_embedding))

print(f"\n🧠 Vector Search Only (similar to headphones embedding)")
print("-" * 60)
for name, cat, price, dist in cur.fetchall():
    print(f"  {name:<40} (distance: {dist:.4f})")

# Method 3: Combined
cur.execute("""
    SELECT name, category, price,
           CASE WHEN search_text @@ to_tsquery('english', %s)
                THEN ts_rank(search_text, to_tsquery('english', %s))
                ELSE 0 END AS text_score,
           1 - (embedding <=> %s::vector) AS vector_score,
           (CASE WHEN search_text @@ to_tsquery('english', %s)
                 THEN ts_rank(search_text, to_tsquery('english', %s)) * 10
                 ELSE 0 END
            + (1 - (embedding <=> %s::vector))) AS combined_score
    FROM products
    ORDER BY (CASE WHEN search_text @@ to_tsquery('english', %s)
                   THEN ts_rank(search_text, to_tsquery('english', %s)) * 10
                   ELSE 0 END
              + (1 - (embedding <=> %s::vector))) DESC
    LIMIT 5
""", (search_tsquery, search_tsquery, query_embedding,
      search_tsquery, search_tsquery, query_embedding,
      search_tsquery, search_tsquery, query_embedding))

print(f"\n🏆 Hybrid: Full-Text + Vector Search")
print("-" * 70)
print(f"{'Product':<40} {'Text':>6} {'Vector':>8} {'Combined':>9}")
print("-" * 70)
for name, cat, price, ts, vs, cs in cur.fetchall():
    print(f"{name:<40} {ts:>6.3f} {vs:>8.4f} {cs:>9.4f}")

conn.close()

print()
print("💡 Hybrid search surfaces results that match keywords AND are semantically similar.")
print("   Products matching both criteria rank highest.")

---

## 📊 BAD → BETTER → BEST Summary

In [ ]:
print("📊 Hybrid Search: BAD -> BETTER -> BEST")
print("=" * 75)
print()
print("❌ BAD: Vector-Only Search")
print("   Ignores business rules (stock, price, category)")
print("   Returns irrelevant results that frustrate users")
print("   No way to enforce hard constraints")
print()
print("✅ BETTER: Pre-Filter + Vector Search")
print("   WHERE clause filters first, then vector ranking")
print("   Respects hard constraints (in_stock, category)")
print("   Problem: strict filters may eliminate good vector matches")
print("   Problem: binary (pass/fail) — no soft preferences")
print()
print("🏆 BEST: Combined Scoring + Partial Indexes")
print("   Hard filters for non-negotiable rules (in_stock)")
print("   Soft scoring for preferences (price, rating, popularity)")
print("   Weighted combination: alpha * vector + (1-alpha) * metadata")
print("   Partial HNSW indexes for fast filtered vector search")
print("   Optional: full-text search for keyword matching")
print()
print("🎯 In interviews, mention:")
print("   'I would combine vector similarity with metadata filters using")
print("    a weighted scoring function. Hard filters in WHERE clauses,")
print("    soft preferences in the scoring. Partial indexes on common")
print("    filter combinations keep queries fast.'")

## 🧹 Cleanup

In [ ]:
# Clean up: remove the search_text column and partial indexes
conn = get_conn()
cur = conn.cursor()

cur.execute("ALTER TABLE products DROP COLUMN IF EXISTS search_text")
cur.execute("DROP INDEX IF EXISTS idx_products_fts")
for cat in ['electronics', 'clothing', 'home', 'books', 'sports']:
    cur.execute(f"DROP INDEX IF EXISTS idx_{cat}_hnsw")

conn.commit()
conn.close()
print("🧹 Cleaned up indexes and columns")

## 📚 Summary

### Key Takeaways

1. **Vector search alone is not enough** for real applications — business rules matter
2. **Pre-filtering** (WHERE + ORDER BY vector) is simple and works for broad filters
3. **Combined scoring** (weighted vector + metadata) gives the best user experience
4. **Partial indexes** speed up filtered vector search dramatically
5. **Full-text + vector search** combines keyword precision with semantic understanding

### Interview Tip

> "For a product search system, I'd use hybrid search: HNSW vector index for semantic similarity, combined with metadata scoring for price/rating/availability. Hard constraints go in WHERE clauses, soft preferences in a weighted scoring function. Partial HNSW indexes on common category filters keep queries fast even at scale."

### The Complete Picture

```
User Query: "wireless headphones under $100"
    │
    ├── Full-text search → keyword matches ("wireless", "headphones")
    ├── Vector search → semantic matches (Bluetooth speakers, earbuds)
    ├── Metadata filters → price ≤ $100, in_stock = true
    │
    └── Combined Scoring → best results surface to the top
```

### What We Covered in This Lab Series

| Notebook | Key Lesson |
|----------|-----------|
| **1. Embeddings & Similarity Search** | BAD linear scan → BETTER IVFFlat → BEST HNSW |
| **2. Vector Indexing Strategies** | How to benchmark and tune IVFFlat vs HNSW |
| **3. Hybrid Search** | BAD vector-only → BETTER pre-filter → BEST combined scoring |